In [2]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

                                 Matriz de confusão

	                    Previsto: Não inadimplente	Previsto: Inadimplente
Real: Não inadimplente	Acertou (Verdadeiro Negativo)	Errou (Falso Positivo)
Real: Inadimplente	    Errou (Falso Negativo)	       Acertou (Verdadeiro Positivo)

Report = Precisão, Recall (de todos os que realmente inadimpliram, quantos o modelo conseguiu identificar?)
e F1-score (Precisão e Recall)

Roc Auc Score = Probabilidade

In [3]:
conexao = sqlite3.connect("../credito.db")
df = pd.read_sql_query("SELECT * FROM clientes_tratados", conexao)
conexao.close()

df.shape

(104804, 13)

## Separando treino e teste
Dividindo a base em 80% para treino e 20% para teste, mantendo a proporção de inadimplentes igual nos dois conjuntos (estratificação), já que a base é desbalanceada.

In [16]:
X = df.drop(columns=["SeriousDlqin2yrs", "faixa_etaria", "renda_informada"])
y = df["SeriousDlqin2yrs"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_treino.shape, X_teste.shape)

#X = tudo que o modelo usa pra prever (as variáveis do cliente, sem o alvo)
#y = só o alvo (SeriousDlqin2yrs — o que queremos prever)

(83843, 10) (20961, 10)


## Treinando a Regressão Logística
Treinando o modelo com os dados de treino, usando class_weight="balanced" para compensar o desbalanceamento da base (6,6% de inadimplentes).

In [ ]:
modelo = LogisticRegression(max_iter=1000, class_weight="balanced")      #número máximo de tentativas
modelo.fit(X_treino, y_treino)

print("Modelo treinado")

Modelo treinado


c:\Users\ryanp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Input X contains NaN. = alguma coluna com valor nulo

In [18]:
X_treino.isnull().sum()

RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

In [15]:
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

#volta la em cima

## Avaliando o modelo
Gerando previsões no conjunto de teste e calculando as métricas: matriz de confusão, classification report e AUC-ROC.

In [19]:
y_pred = modelo.predict(X_teste)
y_proba = modelo.predict_proba(X_teste)[:, 1]

print("Matriz de confusão:")
print(confusion_matrix(y_teste, y_pred))

print("\nClassification report:")
print(classification_report(y_teste, y_pred))

print("\nAUC-ROC:", roc_auc_score(y_teste, y_proba))

Matriz de confusão:
[[14856  4715]
 [  340  1050]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.76      0.85     19571
           1       0.18      0.76      0.29      1390

    accuracy                           0.76     20961
   macro avg       0.58      0.76      0.57     20961
weighted avg       0.92      0.76      0.82     20961


AUC-ROC: 0.8319554994193803
